# 🎓 Re:Learn — Misconception Classifier Training in Google Colab
### Fine-Tuning DistilBERT with Paired Hard Negatives on Cloud GPU

- **Research Backing**: Brown & Altadmri (2017) 100M+ events; Perkins (1986) Bugs vs Slips
- **Zero Setup**: Dataset is pre-bundled into the notebook — zero file uploads required!


In [ ]:
# Step 1: Install Hugging Face & ML libraries
%pip install -q transformers datasets accelerate scikit-learn matplotlib pandas


In [ ]:
# Step 2: Verify GPU Acceleration
import torch
print('PyTorch Version:', torch.__version__)
if torch.cuda.is_available():
    print(f'✅ GPU Accelerated Runtime Active: {torch.cuda.get_device_name(0)}')
else:
    print('ℹ️ Running on CPU runtime (fast for 290 samples).')


In [ ]:
# Step 3: Instant Self-Contained Dataset Loader (Zero-Upload, Never Hangs!)
import base64
import zlib
import io
import pandas as pd

DATASET_B64 = "eJztXXtvIzeS/9+AvwOj/aOtTI9X/dLDWcfIJZvL4Daz2cvcBouBMaAkSuq41a30w7ZyOGA/xH7C/SRXVST7oYdtaWTLsvtwm7G6ySLrV8WqIpss+kMzSbOhCNNPg2goTBHHUfxpKpKEj4V5E0fh+FOUpbMsNad+MojCgZilfhR+8ofmIIpjMVAVk2gQ89QffPLDVMTXQBBKmbHgSQLEptjAb5lI8Onx0c+tlmU2btk5846P/BGjv86Ojxj83ywGCifGyL8WRrNh/jwPU377Z+zWGfPDax74Q5bQQ3aiqzbNMGKqmz++XaK9lvg3ydUZM84NdsXZlKcB77OrOWeTKIX/cP8C3tHLCwaErgTj4ZDHTPORl2NQZxBNp/D6wrw973zF/IQZ2IWOwejVjMd+EoUXQCG5ETGbi+SP0OU37GYyJzjsh3b5JIxC0TTlb/lSDFk/S5mSIxO3M5AKPAyjdOKHY3aSTqJsPEmB/lRAB1SHgAphdcIDENNwzpQ8GcrzK1YRNyDP+iLwxagpQeNjAAKRU7QApDhVyKXRhN2eeyzwQ0F44jv/wvxmBJohcfEM4JyniFOSRjF0Fhq4vSAoHLORApkAirWOj0ZRzHx8G/NwLE4sk1mtpoJGFntzzvzjIwkTPSlQcj12ksPhecQwIF2QsqBsgxjSz4A6oMQCnqQszKZ96PEVz8JEKYTVAo6ykIV84jPO5/wCCHwXiUQRaJleU0qFeRfwhxj6gKhUzQax55qNBZ68ZkXUftH/Ruv01GUnWrLJJBpciaFZiNg6PfWgOLF1gnLTQlRi+mkOsg/hYQbjEVThik94yBIBpLI400JDxnRXkPkrPwXJSe6TC1O+cpo4AkQgcCwnIFwACOpHxJRnNjjI66Nl2pfHR334k8M/p3w2E+EQqmr58BJvUJjZJnMuSzKiZ5eSI8ckOqeDaDY/Ufz0zznDrgHwsgd9Tn/D/yccxSZ7JQuEnE/Z0AcZmfz8o3X5FVaH/+hu9ZpfMd0tdv41k+rXNhtDMWJ8ODzxUzE1oYnx+cdLLSP4pevj66Z8Gos0i0N8mXMK9a1ms/zTbla4v2TpRIQSswoE+kUBxIoevUcyZ/gngIT/BZ36eKlMGhTnWZDCAxBuzAY8CNhMwQaIFVZLXDHQ4z7n8YX5k1TWM+bEQ1kFWwSxS+2V6HSUncLmHM3cbcFXecB9iDNpY1zsFRv5cQLU0TTK6mgbtS29MI2WegjiUy3q8WNS013QsbGAxg2wHm+K1uEhtP9hPhPKSwxAxaMwQIMWDngqQvgfmJkY7AX+U/UVnimJkpH1jCZSViYB2/kti1IY3KBagBgoqxz4//7nv8CEs2EEpEjvUmgdS4RgDrDal1CyxAdAEsBLYqNnKotOXBiO0cyxc0rgdZuyd6V+0Z9YXnbvDTIDgzpB0Wozy2U/1dhVb7BTNrUG/71OmA1/2sTEiMe/UQnsmtUyG760ujcTPxBgo/606IYmPjmhGCwiAzMmrkXchD6O/BBUkwVRNMNut01QHagOptmCbif+UL2jjvvQ0z4o3XAixuBa8QWLsyt0Fugsw6FPLoc8MZgm85eJP5hIbzLlVyANcGuJrAbOYwZqEqG99VNiApx/qDwp/GOVeDlnYdXQgpiMLw05hLEwdJhZDfNdiNaWYyeURmWhT89A4tA8+FWMFgaTqiZ1ICACVpMUC7pvkxkfQLNUTwUaeRuG7H0eUajR2AfrHF8ACj5Y6BlPwXD9t5AUzsBfIgo24ZAwdLkKWOJPClBFEVZLhRFfk/iU6Pr+GGW3zF2uczxU/cWIIgBfs8igQkH6N9W8P1JSpQgJTMmQ3UCQprw/+ZvAv4JB8yu0KQX6vX8LaIggETTaqSlVFFwqKgVxo+zeMMr6gQAjUxHeLQjPBtHNgV9Vwsut7RxBLowSOG2yqCUbAfqbzWZRjKy+YWhM8SVy2TWVMacWkMsUoiMWjWTDklmJAASjmWDf/fnDNx++0TJM/HjEvnv3Xz+oZxdsTi4VX2GkwyYZR5XmSpk4G2XhgBRetQtAAuzqRwKSIo8rIXElJMlvGSjASaghiUWC5h4UHPocahRUKbfkdpDRkpGx2k3JMg5X1aKkpaMiSRi8wSAL0IxibDeJ2Bg8iQkqGhe9B8Loa6UcweL8VIl9yCex0Qm4LA6hIXv7tRYkPGsSb97nzwiq+tpbEVOz09PTBoyqQTQO/d8Fu/bB5IAwzsrhM9rqhKLb60ROAVRUBT4FOp5CuJwyDqFr4M9O2bvShOcM1R6j3Anv+ylL/Rl5olPEn6lwGnyK//tXUP8KArFg9BanTeEYBsSvnIdv+1H06wRNOSkTNHABvRCxP5qz/lx2LqTBK0FrgzchuyyCIALX8B68kYImRMdk4GsDnS5aLZABjMrhIkzatisaDRgKETKKVXje7TEHZv+W+QN05bcwaqbRtQD2TFBk9v6vH4gvXRpAfksd7CyFu0X8ruNd+UvLGbyqWG2osjDXXGVXF+wTqMIYw1t0ImqMnigzBX2AkvTnECYy4B1BnYO5iitloxjdYyyEGjHxwcGDCCwZHbEsgUgp4eC4Em3VKfwl4+LDhBlbsOhPRe0CyEQRlH7Dfs0gyBnRTM+CMAa8c6KiZdOBeHlhukOzLSxTmeacFwN7YZZTDhvaao6j66gpEgslnxgNgj2bYWAgXynbBGM6hoFLVgwDg1kGwuKqDI72/gXoPXo0YrrxAf8+U+85TewEB/8EUUBMIoMY5Jz4o1mPBWNxFFCs+j0Hu09DUj44pzixOtSjsDTV1dNYPdctmMVyqHCjLEELhAYqVgGnIn/GTgDQDMQFsSf4Wtmmn+LAw0EBbqGIOJQuUBE9J8a+/ZG6nENjqC4brB8F6ON89jtgEgHf0r2RAU9V+xjqgDoF2qvZEGSh8A3eHxi5pf7oXCqN14ZNxnakabeoueiAaPxUdd42E5gnoO4lH99al8oVYyWs3SqmejAz9wsWsHF019J4Qnl/ABpME10Bobi0BB+9y7zrNOtFBv7xHuJEiLFkVIXe11Kym1O4t4F9tsycCFJRLucdBAEypikkQyZmHmUQWcCwgzCeJwmOdGBbrT+Aw/khutHFIGRP0Y/SvBflLMAVBlixlfNgax5koXNmKz7k7y15UcSQ2hPzA9ES2DWIDWYChgYomOUYiiMMIjJUOlTbjZgqU1QkpYrRkge9pb5B5EZd8tN5KYJHrRToQq557HMI0S7Md+BQ4yjAGRcfTn0Yv1oFyTQjHSB6wX6ZzHEQ5dxB4ANtkbr1qraC6m8UF1imJEW0JDu/qMUotAdDfzQSsQA3xvoivREQOGIogNaDYgEwznJZ5cJsvJORCsyc8iU/q2Ug17TcwlWIiHKdiMEVPtM4XTQ0bx7xBgYfanerzCXZAAZmsjl7RA3JPQP+2iX+rM5uGQR6z4DDjhp7OPE0OjsadiENuc6+RlxX8aQjCaO9I74UQUlxP7z1SCO1pXSqKjnDTxZbWJTcTDrPQCVxLUc6HdRJPejCbd0a6WLniV2alYcZWlKaD/l7S15yOT05P3Y5bHJ3Eja5T82Ds2gWrF3Zu8IuWPsyepYMM5IBLtSAVOyqZaBFv61MgyRIFJ+BbfC0uyKXbNi7cljSJSO9/UivrfiSkyZUpF35LEVRkdwPdx3SzWsadYuaCZO0ON1cL6/lgHseWtnV1lF+kNHTSvy1pYUkQk8/tcTFDtIL9bnK8nakhYofIrgXHbSLqIJMh93SgQX83Da0kFYDSD2tlGyILvBzKq3EGdwwmdHH/wyMS70sQ68/dheXZugj8/0LMxUab63L6if5ggiOhr4Y45IzjD65LYKIy4946mOBGpWq2RH9oC+quOQ2xdHIw3mZeM6n/UA+3R3w6e6RT2d5pdlau7MCRmKcUkdapd0Uan/Iij0n1hug1SgNOvnqfZMRJUkIV4Dz7+gMv6Dn32H1Pp25qug2dbfdB4qnvQPxtPcoHu+BfHo74NPbI5/tB/LZ2QGfnT3y2blnm5b34F1a0WiEAwM/TSbZdO0Y9OQQzJ1E/uJ905TfUlU3/XAQZODF3uPerDKX5U1dOeUWdjWPQuzuUtPdXRmR7qPZkN4BdtppPXCgLH0a2WKgOPsbKM5D44wDN3yOfY9BcHZtEJynMAjOvdtRrV3zpYONR2bsoYHHoY8/7z4BujsXoPskAnxopHHgExin8zoiZOcQQw+n9zq8m/vQaOXAldC15A7DGJpIUhGfZImITTaOo2xW2gVPv/U+eCxS3QdPr3NdpV/4irbVzdCy4fbypLS3fWVzaou7rH6u/i1vc/8FN3X6uIdSb3gnnAa4P08eKeGaR71NEfGZY53SFvkhygOIJ1gj389Ie+Vo8SnJwbFrcNaDA73FFiza2WabzIWRgQ9seEAv1E+NjNvO9+vS68K49aN0QlUSNo2G/sgXQwlHhVzlYMh3/pBV3jJ8S3qfH1kBdAI/vGJEHndHJnLpWp5cweMhuBxaOiBig7fWx5R47nddt8SoY7LOPXw6rQPl0yvx2TOZ3bqHUbtzoIy2y5rbhvitfR+n9oFy2qmqrnWvSA+V0W5tqddb6l5JC7p4Au8eJbAOUwm8Vq0Ea5XAqwO9O8CpA707wKkEetYDAr3ugdoPdyGitV9opOd5tbqvV/dydGjfH+7bh6rt5dgQFzG9+xh1DpTRbplRF9fb7xvXvQPltFeP67Xjut1acGP3TobcA1X4tkU7BzlAcS3WHbjD49qlQ3exGGbhEDdbFXuw9ME6SSjfrpVTxkbVsTkOfY4CwYuNWyFuus6Pzl00tOgM3F2ZfKJ9YmeY0EaMRIwJYaJrTBZTepv33MhFCGGa2g/ZpdVrLZ7r8kebSj4KdZZvHOmvK5IrSQX7fKJINUuL5UYUl/Y7ApM3fJ4wgZsa5Znogt9SI9WlWLnWi2uxMtGFV95BqflxND9t6kNnF/woUnvhx9X8uFRkF+xISnvhxnupw6hdUbvezrSutxcxdTQ3Nmm+vQt2FKm98NPNjVyPetHdiZVTtPbCUe+FDqRO66UyZuU6SCNhJ2NKkdqHBnbslyoo56Uy5lat4A6N4F5sYMfL+emQJd5N7Kpo7YWjPIqwvN3ZCG9/NqJTtXmWszujZzl74aj7Uo1DryqqncybFKl9CKrbyvlxSP9bO2FI0doLRxBA6FO1huViLkI3T36Dzz8rg6OmTCoM1DFZoiu5/JYXzA2HwE0qblOdXAfPMnK9y3LoqwxfNz7myyWFB9VL1DrT3QcjHZXLkUg6mEMy5xzzzQrK+ENs4wFSzTk8L6RZYpXO4emsc9BtuZrkmZIOcekWGSGBYLN0fJO2lxpvDBxwN7imRqtHeKxPZg2EjgdRIlfUZHbLi8rJOVzyM1wbhYjixz2qqRiLOGfIKYuSklj2HkuUMu9l77mI0i1x3sNi3iMx3iO+vefCt1eWeAfLWe3HEnmHWLfaz4X39pK2F4HA46i75TwX3ju56ZKcG1ZrF7bLKqWzBYpPary6BUsyI661E47sEkdPy1CvrJ9OJTHyrtXTUdmRn4d29lq5KJVyOrsQZVk3nScVZa8cI7WRpSLu27Eo28Sk3XouorRLnEsb+1ghhTKxzyWk6DmFPZLhYW8n9qgUHvaeVondhVHp7nhQuk/LTiX06SJHjxb5dInJ5xL49JYDn0fj3HtenJfCHlmqu5Mx6RRK3H1aJS6Cno70lDsJejolT/m0QU+vt2g1dzKnLlvNJ51T263WkgN8LNevHOAzcf02phLWCXit/OIJ+eRr1qqsTtLTB90+Ieu/XXn7BPWVLie48YOAbpRQ7QlM4p7xAJiMKZl1qyJFESYZ4KhzE9ENDqmIpz7Cn+QM2fqakpwb/PmnxcyuqT+4euBVGioB1ipW8osx1Ds6OowdG0xwH1RCbSdR6Tle/4OXZkiZDXiWYCnUSNVwibvFZTwbd3i9LGm5L40h76Ux1H5pDHUKW5/zIx98hpGQBLY2E7L6Dg1F96VJDUKOeVlic5BWb1tpzbeX1HyHUsIkwWGZqRCYWkwt+XCuwu25CnfJ1UsLKTBv8Mti6KWFEdZLCyMwRbDOoltcB0cPwEK0trUQksLWVkJW36GlKMUWC1IDLq1tuSwuENmKS1l9h1zmG1lyHvHnn1hnWwavZZak7djDyjtk7qUFGpg8eOX8cftR94wmkJhA+EWJC7MIY7qlW7buOt9bOk31gLvNVl4xSXl6oiylNGB5HRZA54PVt59VWpIY/DyJsmCoLzmT96LhNW24WwdBnRfXdpWvJvTD0oWW5dbxFukVsrWXoOi9ViicJSic1wqFuwRFkXDvtWHh1WqhoWgvQdF5rVDIix3Q0X7NrNKFvWM6+ho/0qW9eXJK2RuTTQrfl+DfKTLEpe/siyDCm6aL+wHwVpP8blqde79IbClvGqWwIRlEM0FfXdRNv/oLQh/vkEzY//5fjkR32VZYr1UrerU3VVA4rdJ1XTBEuq94iDjL8eardaeOPMUVgk68apVYDjat1xpWOHW0WWCxHG2+Wm/qtPPLs76WK3uv1lp06hGSY7Ecb75aKJbDzVdrLDDXd72mJ6FQWSKT3zIei5NQw8BD3DUXsi9ZmO9Tk0Usu1nsv6M3CcPcT8hO11QppaC6so1I6IYnmENqkAW4T89kfbBbQ38ok5brDE4T3Drmg0HjYN1SuSCNiadETGdIgb8rMb+J4iFLRDhM2JCnXCdc50XGKM2XvTFf7kHwpVJwYbMnHJqs6i1nb+DR8ZHc9EiF2iZzm6XNk8C6Xd1tmIVJNptFMTo/8DIxdjiF1ydJE1fl2ZdnzEAksIq6HxJoGVVNLiDCLiiQ6AiqbLlJ7OU3RZbR0LcWmqjav2aJdrO42XEQCxFqoEjBC0x0/rL+PM8P5lMm/JiOKEP3VIHCx2oU3U1RtDwTk6nUOFZx9DYdZd5BDLL2pmx1D4KtzqZstQ+Cre7Gpv4wtLC3qZHq0V1HtY2q2CidJPzhMHYBxhrFBRQ3jhMPwnZ4G4eJB2HpPWdjk+gcBF/ui4w3dHruDcPRTm2kFozUxnFb7yDUo7PFnK9dK8eCcmwcJh6Gqe9tylbnENhqbxy1WfVCx7LSt63KZZHjsLzSyZOkYb7nUw0PJiBnBpUyJN1UfYIYViHomQsXUAJNxBnYCPzZGUvhaTKj87KDCY/5IMU7JqG0QcWNU/N9lPoD9WVjJoIAT6NifjH1XvceojPcoO4dHwG/9Fd1i+zIvxZGk/2B/ZxmtA6r09JdhdFNgnnkoFHoQtowf56HKddXNvqhTDSX0MNF5m6xomf+DLyAygxFLHcmnOatzGJx7UdZEqA8plGYAMOoU0M/SX0pslPzb5kPkk9EMHo7mIgBKEkCXAa48fcatD/K+vj3eS4p5xXx6r4iXr1XxGsbHArydCOCQTSl9IIz+txgNJcNDRa9z84ojFbRK9mbqR+CPcpNDYumfoLn1k/Zt1GMp9rpHL2Ypacm3sqAH2eBLFppNED9zA+A5dx8L7mizkHb0O6LlEnv9YyqTuuQ9Q8zi99MAl+Uk9H2QRZXK/pNBe/rd0Oe11gm96DeU+U7eq/e697bB9175yWO/c4rih463ouUYPugR9VBxwOd7kH3/hX5/e6h+v3//PP7Tz+8b1mffoT/tTD1VcrjsUhzuRU/q8KjzVO40+oO0ZyyH/m8L8jk4JamFHPNG7igYpydY/6tPAeYcW5csBPaeWk39YIXIGCZf2B/FzFd4kbX1KHVirN0YjboijqkfMPlsgzYyBnm/0pvIrmekzDMiZUlDATgj0PKoqVWevxQZVO/NSu7OLNEgYUiSubTfhRU1nmMWzorS7nHFHGDFA6flTdsVXD9e0tCqzPNSGTzX/sCFhuYiMT/HbC91iBPsxToHRa8SnPluXoJrvq7hvYzobUltH4qpjm2+kcN7meC60hw9Rl8iW7+q4b3M+F1Jbx3xiA1rJvB2gNMLbOxHLNlGIVA3YGcc0BwxfvRtTg+0m4OM0wUXu98OYVGNJMvjebx0b6DkHZZer21Qcg/gCYxTiRnIh5J1nEDOZJxvmAfMMcFZ79RPDkOOO4K10kw2v/+578gfkullKY8CPATV3RhNmT4qSJPTa1t0pe80gRO6gJ+REJ+pM74CcissSA0ikCeQG77jXCWpHbXmDsc2VlPI7v9hU8vVG7208htn7HZC5Wc8zSS22/g90Jl5z6N7PYTVb40mdkgszYIzF6R+9QrQxougpkfBzbOjFUhd3v9ylHpCPANpU3Ll85TEQQycTqyEM9iQU/lueBQ3CwcPMbFuwRi0xSicYqscZMPhda0oSoKABt5kBgqnjGDVvPQQENwjf/qbUj4d9O4WECFYrbHAebOucihwGPV8NwFj13Dcxc8Tg3PXfC4NTxr4cE1EKfyvWdF8jjjL5ij0jozTOY31yIoS9lGUz6USbvvB9V58PJEzEZ+nKQyY+aEy2++sT+epGyGX3ozOnD+BfsWnTm9RM8NQccc6+qMoOqy4eIDMh1QTyM8wR5kmB9DQcrTMg38UyXshF4vgkj+7ZnheE849ZzRtGo0d4imXaO5QzSdGs0dounWaO4CTQfQ9ABK12wAGANcAcA7PqfJGC/++hClPIAoAmID9oZRAX1eBUpA4NN4yB1gJ7i3o9GAWo1GU90I1ihh56313d+VPuSkmBer2iWutm+pC8HsFsy2i4892IdxFM+RfTy7c2H+FIuhP0iL++GByF/joYglf17+Aaj0EPp64jUX8CKvXbp7ey1kVOCRILtL3Z4tcJYETi6tr4UNXtegFaDZEjS1rL0WNXxfw1bA5kjY9JLyWtyoQA1cAZwrgbu9C7TbGjAFGM6EPbORUk8kYgGHaXWOWVGTyqivVXgO1Dk+SrLplMd4kZfxDp8WQznZEap3f7m/iflshnEN9c2gNI3U12a+qG1/wXSwAeEW7kGgFQdZM8ZiBvXXwLRueH1HfgxUr4XjJoycmtzZgVd1pkpUdJdGiS+/vOHCKebJzx3l+75DHArWVo31k2Ft11g/GdZOjfWTYe3WWD8+1i5gjQakbTagNGL30WqZALbJnNbl8dHC6oiFCScqCyRY66N/qXLF6sWOwE9kKtZbne+C6psNlKLTaiB+9tqoonz1tiIi1x8oB6xcAxEBZX3ArmHqWRXTYbsXZuN76DWXnYCagQjHALurPobgflD8Ci93Efjy1nLVzixKaHEkKX1clxBR+AAoBf7UTzeHiaptj9NdevYs0bIkWlN++0nO1DfES1V8RYipUdjHcSAN14aQ6ZqvCDNHYqZdxIaAUbVXhJYr0dpCtV6NTuFMvCPXyZZ9oVrr/576hUXoqwAW/di6zNMwGX/hS6+dh6JXos2g7Tvn3g0MMIpdeOMozTvzBXunkyUhCo58oXBZBbgTD6nMBcRG7whJ+mSfNLG0U60IptmfZtMcVyCzhCJ5y+cH5J2bJp4znFYN5y7htGs4dwmnU8O5SzjdGs6dwOkBnF3AslvkVaR0jp+gI/zkxmSTapRzw76ER8dHx0dYQCVZLNWAwAIX7kd+yINPgyjBaSCV/JL2vCvkv4UXCHpRbDcZGe2WTMV490n+UgZFmegSAats2otFgtkUxxHuOVdZGmUqRsrRuDKXJQqiRBlphUaan4ykFJeGTMhoKAnJM5XyfhZozb8u8kLeTCK8zUWSR2EvyYzChyWx0Rz6TrnpyflinVWSk2WfWHT3n159IQK0VglQTervFGGxYrBcb5UYdflakI8jSHuVIPVSw52SLK1krKi5SpZ5jVqYjyNMZ5UwaRnkTknq9ZXFOqtkKMvWAnwcAbqrBHj3MKyOv9UDrx5xjyIwXE1SucrHIv2U8tsToKYFpRJYS+fVOrVQWLuMU1XacCzyBh5sJME3d0pw/ZqUSpSluEVckxzP4juX3nCKsOu3lZOa1HV6hLeGVtm7MBsfIplwHEVG/C3L2sS7SPEywWplalJmIAcuZdNLQqPo83Pk9tmBaumb0WPI7q7R9zIkaH2mBHcUqVa+ZdWS3EaS9mdKcnehavUrWy3NbaTpfKY0PztWLX35qyW4jQTdz5TgVsGq+gpZS+zhEvvxfz6AwJxP33xqWS2zEcX+GKN5Wk82mW0y5xLsHrSUzeCZfq0fneLmo3B4Utwuo0sUF+k0NCGTuZf0vddZvziJYBlLzRlsQHfVQ0yeCDyZlGePpcVpiuMp4uYs9AdXlPFVASNuqeRYfgrWkT1w511+xfq4Mgv/aDbaTRXMD4tLa3hTbdteRgwDwCcF7c7p0GFBZ9XQbQudXUO3LXRODd220Lk1dFtB9x+Am7ohVlaHQJO+VJoqz+YnbPL846WOdUpPdYOUFqMSOZUK5Ss4OXWDG83misd9o1lGHYqB6PAfE8A07vXN7/DiNX3LGrSf0YYr+joscR+yUSySCRO0Lyv1p3K1LF9G8xO1BEaSoM3UeFUCLtiFOiV7JWUqLdVJSQ4oIVbEquzoYGYRcvLMa1Cnk7KLoBcP12NelNk95HenaTkg4K31wEO0vwi7frQedF2ihnwt5PZ6yPG/i5jnz9aDnhepUV+LurMedTrmuwh78XA97kWZGvi1wLvrgb9dBP32HsBva7DXgu3SmoCtUyi0KFfmCKpj3GVocK7pQNUo4HRyeyKCIFILKDdRHAzzcliiBJ2mc3xExQg490E5JIcRM4C+aiSKjfwYk0rc7uNHPrzu6I/f8yCRN8oOKcqb59ETLqFgHpRArwZpuNSGMXELrwF/oEgHoejItoeUrNYSSBRsPDFOD03r9gzRsmq0NkDLrtHaAC2nRmsDtNwarYeh1SFXuCYJmMpcuTY5Jw8Xk2XyESbTxCSYRnEveX4011qVyrOz1jf+EN1UriGJwc+PQwAYs3ANJkQzKZJv0rcFLlODUW8uiMIUD00lMz6AsuobQl8U/c7rqUag629LJBaxIo+4b7juUrVnCZpVg7Y5aHYN2uagOTVom4Pm1qBtCBoufbtqD0AsRHqCK+4FXsYPGE1Qvg16sSl6xfZMnPL60oNv50RlVECN8HzHZwHkb5kfC9ookMsqKaBA/vJ7fOnkVCgzYvaj4ZxiCiyRiiQ9aZ7R/aSLMJG/fG5I3X8ie394WTVeG+Fl13hthJdT47URXm6N1914/T+6P2Ae"

csv_bytes = zlib.decompress(base64.b64decode(DATASET_B64))
with open('misconceptions_expanded.csv', 'wb') as f:
    f.write(csv_bytes)

df = pd.read_csv('misconceptions_expanded.csv')
print(f'✅ Successfully loaded {len(df)} verified student samples into Colab!')
print('Taxonomy breakdown:')
print(df['misconception_id'].value_counts())


In [ ]:
# Step 4: Multi-Modal Context Representation & 75/25 Split
from sklearn.model_selection import train_test_split
from datasets import Dataset  # type: ignore

df['text'] = (
    df['student_code'].astype(str) + 
    ' [ERR] ' + df['error_message'].fillna('(none)').astype(str) + 
    ' [OUT] ' + df['wrong_output'].fillna('(none)').astype(str)
)

labels = sorted(df['misconception_id'].unique())
lab2id = {l: i for i, l in enumerate(labels)}
id2lab = {i: l for l, i in lab2id.items()}
df['label'] = df['misconception_id'].map(lab2id)

train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df['label']
)
print(f'Training samples: {len(train_df)} | Held-Out Test samples: {len(test_df)}')


In [ ]:
# Step 5: Tokenize with DistilBERT
from transformers import AutoTokenizer  # type: ignore

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=256
    )

train_ds = Dataset.from_pandas(train_df).map(tokenize_batch, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize_batch, batched=True)


In [ ]:
# Step 6: Fine-Tune DistilBERT Misconception Classifier
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer  # type: ignore
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2lab,
    label2id=lab2id
)

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    return {'accuracy': acc, 'macro_f1': f1}

training_args = TrainingArguments(
    output_dir='./colab_checkpoints',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    report_to=[],
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

print('Training started on Colab runtime...')
trainer.train()


In [ ]:
# Step 7: Classification Report & Dark-Mode Confusion Matrix
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

preds = trainer.predict(test_ds)
y_pred = preds.predictions.argmax(-1)
y_true = preds.label_ids

target_names = [id2lab[i] for i in range(len(labels))]
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(9, 8), dpi=150)
fig.patch.set_facecolor('#0b0f19')
ax.set_facecolor('#0b0f19')

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Blues', ax=ax, xticks_rotation=45, colorbar=True)

plt.title('Re:Learn IDE — DistilBERT Misconception Classifier\n(Held-Out Test Set with Paired Hard Negatives)', pad=15)
plt.tight_layout()
plt.savefig('confusion_matrix_distilbert.png', facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# Step 8: Export Production Model
model.save_pretrained('./relearn_distilbert_prod')
tokenizer.save_pretrained('./relearn_distilbert_prod')
print('✅ Exported production weights to ./relearn_distilbert_prod')
